# Answer Quality Filter: Training and Evaluation

**Objective:** Train a DeBERTa-v3-small binary classifier on labeled_asqa.csv
to accept correct answers and reject hallucinated ones, then evaluate against
a no-filter baseline on a held-out test set.

In [ ]:
import sys, json, logging
import numpy as np
import pandas as pd
sys.path.append("..")

from src.filtering.data_split import load_and_split
from src.filtering.learned_filter import AnswerQualityClassifier, train_classifier
from src.filtering.filter_evaluator import FilterEvaluator

logging.basicConfig(level=logging.INFO)
SEED = 42

## 1. Data Split

In [ ]:
train_df, val_df, test_df = load_and_split("../data/asqa/labeled_asqa.csv", seed=SEED)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Label distribution (train): {train_df['label'].value_counts().to_dict()}")

## 2. Train Classifier

In [ ]:
from pathlib import Path

model_dir = Path("../models/answer_filter")
if model_dir.exists() and (model_dir / "config.json").exists():
    print(f"Model already exists at {model_dir}, skipping training.")
else:
    model_dir = train_classifier(
        train_df, val_df,
        output_dir=str(model_dir),
        config_overrides={"batch_size": 4},  # use 16 on GPU
    )

## 3. Threshold Tuning (Validation Set)

In [ ]:
clf = AnswerQualityClassifier(str(model_dir))
evaluator = FilterEvaluator()

val_decisions = clf.predict_batch(val_df["question"].tolist(), val_df["answer"].tolist())
val_confidences = [d.confidence for d in val_decisions]
val_labels = val_df["label"].tolist()

sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in val_confidences]
    r = evaluator.evaluate(preds, val_labels)
    sweep.append({"threshold": round(float(t), 2), "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
best_threshold = best_row["threshold"]
print(f"Best threshold: {best_threshold} (F1={best_row['f1']:.4f})")
sweep_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", marker="o")
ax.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall", marker="s")
ax.plot(sweep_df["threshold"], sweep_df["f1"], label="F1", marker="^", linewidth=2)
ax.axvline(best_threshold, color="gray", linestyle="--", label=f"Best t={best_threshold}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Tuning (Validation Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Final Evaluation (Test Set)

In [ ]:
test_decisions = clf.predict_batch(test_df["question"].tolist(), test_df["answer"].tolist())
test_preds = [d.confidence >= best_threshold for d in test_decisions]
test_labels = test_df["label"].tolist()

learned_result = evaluator.evaluate(test_preds, test_labels)
baseline_result = evaluator.compute_no_filter_baseline(test_labels)

comparison = evaluator.compare(
    {"No Filter": baseline_result, "Learned Filter": learned_result},
    save_path="../results/filter_comparison.json",
)
pd.DataFrame(comparison).set_index("strategy").round(4)

## 5. Error Analysis

In [ ]:
test_df = test_df.copy()
test_df["predicted"] = test_preds
test_df["confidence"] = [d.confidence for d in test_decisions]

fp = test_df[(test_df["label"] == 0) & (test_df["predicted"] == True)]
fn = test_df[(test_df["label"] == 1) & (test_df["predicted"] == False)]
print(f"False Positives (hallucinations accepted): {len(fp)}")
print(f"False Negatives (correct answers rejected): {len(fn)}")
print(f"\nSample False Positives:")
for _, row in fp.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")
print(f"\nSample False Negatives:")
for _, row in fn.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")

## 6. Apply to Real RAG Outputs

In [ ]:
rag_df = pd.read_csv("../results/asqa_normal_rag_predictions.csv")
rag_decisions = clf.predict_batch(rag_df["question"].tolist(), rag_df["predicted_answer"].tolist())
rag_df["filter_accept"] = [d.accept for d in rag_decisions]
rag_df["filter_confidence"] = [round(d.confidence, 4) for d in rag_decisions]

n_accept = rag_df["filter_accept"].sum()
print(f"Accepted: {n_accept}/{len(rag_df)} ({100*n_accept/len(rag_df):.1f}%)")
print(f"Rejected: {len(rag_df)-n_accept}/{len(rag_df)} ({100*(len(rag_df)-n_accept)/len(rag_df):.1f}%)")

rag_df.to_csv("../results/rag_predictions_filtered.csv", index=False)

## 7. Ablation: Training Data Size

Train on 25%, 50%, 75%, 100% of the data to measure how much labelled data the filter needs.

In [ ]:
fractions = [0.25, 0.50, 0.75, 1.0]
data_size_rows = []

for frac in fractions:
    tag = f"data_{int(frac * 100)}pct"
    n_samples = int(len(train_df) * frac)
    subset = train_df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag} ({n_samples} samples)...")
        train_classifier(subset, val_df, output_dir=str(out_dir), config_overrides={"batch_size": 4})

    abl_clf = AnswerQualityClassifier(str(out_dir))
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["question"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["question"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    data_size_rows.append({"fraction": frac, "n_train": n_samples, "threshold": best_abl_t,
                           "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ds_df = pd.DataFrame(data_size_rows)
ds_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ds_df["fraction"] * 100, ds_df["f1"], marker="o", label="F1", linewidth=2)
ax.plot(ds_df["fraction"] * 100, ds_df["accuracy"], marker="s", label="Accuracy")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Score")
ax.set_title("Ablation: Training Data Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Ablation: Max Sequence Length

Compare max_length 256, 384, 512 to measure the trade-off between speed and accuracy.

In [ ]:
max_lengths = [256, 384, 512]
ml_rows = []

for ml in max_lengths:
    tag = f"maxlen_{ml}"
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag}...")
        train_classifier(train_df, val_df, output_dir=str(out_dir),
                         config_overrides={"batch_size": 4, "max_length": ml})

    abl_clf = AnswerQualityClassifier(str(out_dir))
    abl_clf.max_length = ml
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["question"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["question"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    ml_rows.append({"max_length": ml, "threshold": best_abl_t,
                    "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ml_df = pd.DataFrame(ml_rows)
ml_df

## 9. Results Summary

In [ ]:
print("=== FINAL RESULTS ===")
print(f"Model: microsoft/deberta-v3-small")
print(f"Trained on: {len(train_df)} samples, Evaluated on: {len(test_df)} samples")
print(f"Best threshold: {best_threshold}")
print(f"\nNo Filter  -> P={baseline_result.precision:.3f} R={baseline_result.recall:.3f} F1={baseline_result.f1:.3f} Acc={baseline_result.accuracy:.3f}")
print(f"Learned    -> P={learned_result.precision:.3f} R={learned_result.recall:.3f} F1={learned_result.f1:.3f} Acc={learned_result.accuracy:.3f}")
print(f"\nRAG outputs: {n_accept}/{len(rag_df)} accepted ({100*n_accept/len(rag_df):.1f}%)")